# 구조공학 서버 — 첫 번째 단계: 도입과 첫 도구 만들기

> [!ref] 강의노트 매핑
> - **강의노트 §2.7 단계 ①** — 구조공학 도메인 서버 도입부에 해당합니다.
> - **선행 학습**: 첫 번째부터 세 번째 노트북까지 다룬 모델 컨텍스트 프로토콜 기초, 시각적 검증 도구 사용법, 클라이언트 구현을 미리 익혀 두면 좋습니다.

## 학습 목표
이 노트북에서는 한국 건축공학 도메인의 핵심 설계기준인 **한국 콘크리트구조 설계기준** 을 다루는 첫걸음을 내딛습니다. 구체적으로 다음 네 가지를 달성합니다.

첫째, 도메인 진입을 위해 한국 콘크리트구조 설계기준의 영역에 들어가, 본 모델 컨텍스트 프로토콜 라이브러리를 한국 건축공학 도메인 전용 서버로 초기화합니다. 둘째, 첫 도구로 가장 단순한 검토 함수 한 개를 만들어, 철근 보강이 없는 무근 콘크리트 기둥의 압축강도를 검토하는 절차를 노출합니다. 셋째, 단위 일관성을 위해 메가파스칼과 밀리미터와 킬로뉴턴으로 구성된 국제 단위 체계를 강제하고, 자동 스키마 검증을 활성화하여 모델이 인자를 정확히 채울 수 있도록 보장합니다. 넷째, 누적 빌드업의 출발점으로 작성한 서버를 파이썬 모듈 파일로 내보내, 다음 단계 노트북에서 리소스를 덧붙이는 시작점으로 삼습니다.

## 사전 준비 사항
파이썬 환경에 모델 컨텍스트 프로토콜 명령줄 도구와 파이단틱 패키지가 설치되어 있어야 합니다. 첫 번째 트랙의 도구 정의 노트북에서 다룬 데코레이터 패턴을 미리 학습해 두기를 권장합니다. 본 노트북은 그 위에 한국 설계기준 도메인 지식을 한 겹 얹는 구조로 설계되어 있어, 데코레이터 기본기를 갖춘 학생이라면 자연스럽게 따라갈 수 있습니다.


## §1. 왜 구조공학 도메인 전용 서버가 필요한가?

표준 예제인 일반 문서 관리 서버는 **범용 도메인**, 즉 텍스트 파일을 다루는 일반 시나리오를 보여 줍니다. 좋은 출발점이지만 한국 건축공학 현업에서 곧바로 쓸 수 있는 형태는 아닙니다. 우리가 만들 구조공학 서버는 한 단계 더 나아가, **한국 건축공학 도메인의 자산을 거대언어모델에 직접 노출** 하는 것을 목표로 삼습니다.

도메인 자산을 어떤 형태로 노출할지 정리해 보겠습니다. 첫째, 한국 콘크리트구조 설계기준의 조문은 리소스 형태로 노출하여 모델이 검토 직전에 컨텍스트로 첨부할 수 있게 합니다. 둘째, 콘크리트와 철근의 등급별 물성치 값은 룩업 테이블로 노출하여 모델의 환각을 방지합니다. 셋째, 휨 강도와 전단 강도를 계산하는 검토식은 도구로 노출하여 모델이 직접 호출할 수 있게 합니다. 넷째, 마이다스 시빌의 입력 파일을 읽어 절점과 부재 정보를 추출하는 파서는 도구로 노출하여 구조해석 결과를 자동으로 검토할 수 있게 합니다. 다섯째, 표준 검토 시나리오는 프롬프트 템플릿으로 노출하여 사용자가 슬래시 명령으로 호출할 수 있게 합니다.

| 도메인 자산의 종류 | 노출되는 형태 | 한국 건축공학에서의 구체적 예시 |
|---|---|---|
| 한국 콘크리트구조 설계기준 조문 | 리소스 | 콘크리트구조 설계기준 전체 요약, 휨 부재 조문, 전단 부재 조문 |
| 콘크리트와 철근의 재료 물성치 룩업 | 리소스 | 콘크리트 등급별 탄성계수와 인장 균열강도, 이형철근 규격별 단면적 |
| 구조 검토 계산 도구 | 도구 | 휨 강도 검토 함수, 전단 강도 검토 함수 |
| 구조해석 입력 파일 파서 | 도구 | 마이다스 시빌의 입력 파일에서 절점과 부재를 추출하는 파서 |
| 표준 구조 검토 시나리오 | 프롬프트 | 콘크리트구조 설계기준 인용 휨과 전단 종합 검토 템플릿 |

> [!finding] 핵심 통찰: 도구와 리소스와 프롬프트의 제어 주체 차이
> 모델 컨텍스트 프로토콜의 세 가지 노출 형태는 **누가 호출 시점을 결정하는가** 라는 관점에서 명확히 구분됩니다. 도구는 거대언어모델이 제어합니다. 즉 모델이 "지금 이 함수를 부르면 좋겠다"고 스스로 판단해 호출하는 능동적 자산입니다. 리소스는 클라이언트 애플리케이션이 제어합니다. 사용자나 앱이 명시적으로 "이 컨텍스트를 첨부하라"고 지시할 때 주입되는 수동적 자산입니다. 프롬프트는 사용자가 제어합니다. 사용자가 슬래시 명령이나 메뉴를 통해 미리 정의된 표준 템플릿을 호출하는 방식입니다. 도메인 서버의 진가는 이 세 가지를 한국 설계기준에 맞춰 **유기적으로 결합** 하여, 각 자산이 가장 자연스러운 자리에 놓이도록 설계하는 데 있습니다.

> [!ref] 강의노트 §2.7 인용
> "구조공학 도메인 응용 서버는 여섯 단계로 빌드업한다. 먼저 휨과 전단 검토 도구를 정의하고, 다음으로 한국 설계기준과 재료 물성치를 리소스로 등록하며, 그다음 다섯 번째 주차에서 만든 검색 보강 생성 체인을 통합하고, 이어서 마이다스 입력 파일 파서를 추가하고, 시각적 검증 도구로 점검한 뒤, 마지막으로 명령줄 도구에 등록하는 순서이다."


## §2. 환경 설정 — FastMCP와 Pydantic 라이브러리 가져오기

본격적인 도구 정의에 앞서, MCP 서버를 만드는 데 필요한 핵심 라이브러리 두 가지를 가져옵니다. `FastMCP`는 데코레이터 기반으로 도구·리소스·프롬프트를 손쉽게 등록할 수 있게 해 주는 고수준 추상화이며, `Field`는 함수 인자에 설명을 붙여 LLM이 정확히 어떤 값을 넣어야 하는지 알 수 있게 해 줍니다.


In [ ]:
# Stage 1: structural-mcp 초기화
# Week_07.md §2.7 단계 ① — 도메인 MCP 첫 진입
import json
import math
from mcp.server.fastmcp import FastMCP
from pydantic import Field

print("✅ FastMCP + Pydantic 로드 완료")


## §3. 서버 인스턴스 만들기 — 카멜케이스 이름과 케밥케이스 식별자

이제 서버 객체를 생성합니다. 이름과 표기 규약은 다음과 같이 통일합니다.

서버 이름은 카멜케이스로 표기하며, 시각적 검증 도구나 명령줄 도구 화면에서 사용자에게 보여지는 표시명으로 사용합니다. 명령줄 식별자는 케밥케이스로 표기하며, 나중에 명령줄 도구에 등록할 때 사용할 등록 키 역할을 합니다. 명령줄 도구는 일반적으로 소문자 하이픈 표기를 선호하기 때문입니다. 로그 레벨은 오류 수준으로 지정하여, 학습용 노트북 환경에서 표준 출력 노이즈를 최소화합니다. 운영 환경에서는 정보 또는 디버그 수준으로 올려 자세한 추적 로그를 받을 수 있습니다.

도메인 명명 규약을 일관되게 유지하는 것은 여러 도메인 서버를 동시에 운영할 때 혼선을 줄여 주는 작은 그러나 중요한 습관입니다. 특히 한 사람의 학생이 여러 서버를 만들고 명령줄 도구에 동시에 등록하는 경우, 일관된 명명 규약이 없으면 어떤 서버가 어떤 도메인을 담당하는지 헷갈리기 쉽습니다.


In [ ]:
# Week_07.md §2.7 line ~1505 — FastMCP 초기화 패턴
mcp = FastMCP("StructuralMCP", log_level="ERROR")
print(f"서버 이름: {mcp.name}")


## §4. 첫 번째 도구 정의 — 무근 콘크리트 기둥의 압축강도 검토 함수

검토 대상은 철근 보강이 들어가지 않은 무근 콘크리트 기둥의 축방향 압축강도입니다. 한국 콘크리트구조 설계기준에서 무근 부재의 압축에 적용하는 강도감소계수는 영점 육오를 사용합니다.

핵심 수식은 다음 세 가지입니다. 첫째, 공칭 압축강도는 콘크리트 설계기준 압축강도와 총 단면적의 곱에 영점 팔오를 곱한 값으로, 무근 콘크리트가 이론적으로 견딜 수 있는 최대 압축력을 의미합니다. 둘째, 설계 압축강도는 위 공칭 압축강도에 강도감소계수 영점 육오를 곱하여 안전여유를 확보한 값입니다. 셋째, 소요 대 내력 비율은 외부에서 작용하는 소요 축력을 설계 압축강도로 나눈 값으로, 일점 영 미만이면 안전, 일점 영을 초과하면 위험으로 판정합니다.

> [!tip] 인자 설명 데코레이터의 역할 — 도구 품질의 절반은 설명에서 결정됩니다
> 인자에 설명을 붙이면 본 라이브러리가 이를 자동으로 도구 명세에 포함시킵니다. 거대언어모델은 도구를 호출하기 전에 이 명세를 읽고 어떤 인자에 어떤 값을 넣을지 결정하므로, **설명 문장의 품질이 곧 도구 호출 정확도** 를 결정합니다. 설명에는 단위와 물리적 의미를 명시적으로 적는 것이 좋습니다.


In [ ]:
# Week_07.md §2.7 — 도메인 첫 도구 (간단 사례 시작)
# 기존 백업 S6_07.ipynb에는 휨/전단 도구만 있었음 — 이 노트북은 더 단순한 압축 검토부터 시작
@mcp.tool()
def check_concrete_strength(
    fck: float = Field(description="콘크리트 설계기준 압축강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Ag: float = Field(description="기둥 총 단면적 (mm²)"),
) -> str:
    """무근 콘크리트 기둥의 압축강도를 KDS 41 17 00 기준으로 검토합니다.

    Returns JSON with:
        Pn      : 공칭 압축강도 (kN)
        phi_Pn  : 설계 압축강도 (kN)
        DCR     : 소요/내력 비율 (1.0 미만이면 OK)
        check   : OK/NG 판정
    """
    # KDS 41 17 00 — 무근 콘크리트 압축 강도감소계수
    phi = 0.65

    # 공칭 압축강도 (kN 단위로 변환: MPa × mm² = N → /1000 → kN)
    Pn = 0.85 * fck * Ag / 1000.0
    phi_Pn = phi * Pn
    DCR = Pu / phi_Pn if phi_Pn > 0 else float("inf")

    result = {
        "fck_MPa": fck,
        "Ag_mm2": Ag,
        "Pn_kN": round(Pn, 2),
        "phi_Pn_kN": round(phi_Pn, 2),
        "Pu_kN": Pu,
        "DCR": round(DCR, 3),
        "check": "OK" if DCR <= 1.0 else "NG",
        "reference": "KDS 41 17 00 (φ=0.65 무근)",
    }
    return json.dumps(result, indent=2, ensure_ascii=False)


print("✅ 도구 1개 등록: check_concrete_strength")


## §5. 자동 스키마 검증 — `mcp.list_tools()` 호출

FastMCP는 도구를 등록할 때 Pydantic의 `Field` 정보를 자동으로 읽어 도구 명세(JSON Schema)를 만들어 둡니다. 이 명세는 LLM이 도구를 호출할 때 어떤 인자가 필요하고, 각 인자가 어떤 의미를 가지는지 알려 주는 핵심 정보입니다. 따라서 **설명 문장의 품질이 곧 도구의 호출 정확도** 를 결정합니다.

여기서는 `mcp.list_tools()` 비동기 메서드를 호출하여, 방금 등록한 도구가 어떤 명세로 노출되는지 확인합니다. 이 단계는 운영 환경에서도 도구 배포 직후 반드시 수행해야 하는 검증 절차입니다.


In [ ]:
# Week_07.md §2.7 — 도구 명세 검증
import asyncio

async def show_tools():
    tools = await mcp.list_tools()
    for t in tools:
        print(f"📦 {t.name}")
        print(f"   설명: {t.description.strip().splitlines()[0] if t.description else ''}")
        print(f"   스키마 properties: {list(t.inputSchema['properties'].keys())}")

await show_tools()


## §6. 도구 호출 시연 — 폭 삼백 밀리미터 정사각형 단면 기둥 사례

이제 정의한 도구가 실제로 정확한 결과를 내는지 간단한 사례로 확인합니다.

입력 조건은 다음과 같이 설정합니다. 콘크리트 설계기준 압축강도는 이십칠 메가파스칼로 한국 건축물에서 가장 흔하게 쓰이는 보통강도 콘크리트입니다. 기둥 총 단면적은 구만 제곱밀리미터로 한 변이 삼백 밀리미터인 정사각형 단면입니다. 소요 축력은 오백 킬로뉴턴으로 외부에서 작용하는 압축력을 가정합니다.

> [!action] 손계산으로 미리 예상해 보기
> 공칭 압축강도는 영점 팔오 곱하기 이십칠 곱하기 구만을 천으로 나눈 값으로 약 이천육십오 점 오 킬로뉴턴이 됩니다. 여기에 강도감소계수 영점 육오를 곱하면 설계 압축강도는 약 천삼백사십이 점 육 킬로뉴턴이 됩니다. 따라서 소요 대 내력 비율은 오백을 천삼백사십이 점 육으로 나눈 약 영점 삼칠 정도로 일점 영보다 한참 작으므로 검토 결과는 **안전** 으로 나와야 정상입니다.


In [ ]:
# Week_07.md §2.7 — 첫 도구 호출 검증
result = await mcp.call_tool(
    "check_concrete_strength",
    {"fck": 27, "Pu": 500, "Ag": 90000},
)
# result는 (TextContent[], metadata) 튜플
print(result)


## §7. 파이썬 모듈 파일로 내보내기 — 누적 빌드업의 시작점

지금까지 노트북 안에 정의한 서버를 파이썬 모듈 파일로 디스크에 저장합니다. 이 파일은 다음 단계 노트북에서 한국 설계기준 요약 리소스나 재료 물성치 리소스를 덧붙일 때 출발점이 됩니다. 이렇게 단계마다 디스크 산출물을 명확히 분리하면 학습의 진행 상황이 파일 시스템 변경 사항만 보아도 추적 가능해집니다.

> [!finding] 누적 빌드업 패턴 — 각 단계가 직전 단계 위에 한 겹씩 쌓인다
> 본 트랙의 모든 단계 노트북은 직전 단계가 만든 파이썬 모듈 파일을 입력으로 받아 기능을 한 겹씩 덧붙이는 누적 빌드업 패턴을 따릅니다. 단계별 누적 결과는 다음과 같습니다. 첫째 단계는 도구 한 개를 만듭니다. 이는 지금 학습하는 본 노트북에 해당하며, 무근 콘크리트 압축강도 검토 함수가 그 도구입니다. 둘째 단계는 위에 정적 리소스 네 개를 추가합니다. 한국 설계기준 요약과 콘크리트와 철근 표 그리고 조문 템플릿이 그 넷입니다. 셋째 단계는 정적 조문 템플릿을 다섯 번째 주차의 검색 보강 생성 체인으로 교체합니다. 넷째 단계는 마이다스 시빌의 입력 파일 파서를 도구로 추가합니다. 다섯째 단계는 시각적 검증 도구로 서버를 점검합니다. 마지막 여섯째 단계는 명령줄 도구에 최종 등록합니다.


In [ ]:
# structural_mcp.py — Stage 1 출력
STAGE1_SOURCE = '''"""structural-mcp — Stage 1 (도구 1개)

자동 생성: S6_st01_intro_structural_mcp.ipynb
다음 단계 S6_st02에서 KDS·재료 리소스가 추가됩니다.
"""
import json
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("StructuralMCP", log_level="ERROR")


@mcp.tool()
def check_concrete_strength(
    fck: float = Field(description="콘크리트 설계기준 압축강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Ag: float = Field(description="기둥 총 단면적 (mm²)"),
) -> str:
    """무근 콘크리트 기둥의 압축강도를 KDS 41 17 00 기준으로 검토합니다."""
    phi = 0.65
    Pn = 0.85 * fck * Ag / 1000.0
    phi_Pn = phi * Pn
    DCR = Pu / phi_Pn if phi_Pn > 0 else float("inf")
    return json.dumps({
        "fck_MPa": fck,
        "Ag_mm2": Ag,
        "Pn_kN": round(Pn, 2),
        "phi_Pn_kN": round(phi_Pn, 2),
        "Pu_kN": Pu,
        "DCR": round(DCR, 3),
        "check": "OK" if DCR <= 1.0 else "NG",
        "reference": "KDS 41 17 00 (φ=0.65 무근)",
    }, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("structural_mcp.py", "w", encoding="utf-8") as f:
    f.write(STAGE1_SOURCE)

print(f"✅ structural_mcp.py 저장 완료 ({len(STAGE1_SOURCE)} bytes)")


## §8. 다음 단계 안내 — 두 번째 단계 노트북으로 이어집니다

> [!action] 다음에 학습할 노트북
> 두 번째 단계 노트북에서는 본 노트북에서 만든 도구 위에 두 가지 표준 자원 식별자 스킴을 사용한 리소스 네 개를 등록합니다. 정적 사전 기반 한국 콘크리트구조 설계기준 조문 요약, 콘크리트 강도 등급별 물성치 표, 이형철근 규격별 단면적 표, 그리고 매개변수가 들어가는 동적 리소스 템플릿을 다룹니다. 첫 단계에서 만든 도구를 그대로 둔 채 위에 한 겹씩 쌓아 올리는 누적 빌드업 패턴을 직접 체험할 수 있습니다.

> [!ref] 단계별 학습 로드맵 — 큰 그림으로 보는 여섯 단계
> 본 트랙은 다음 순서로 진행됩니다. 각 단계는 약 삼십 분에서 한 시간 정도 소요되며, 직전 단계에서 만든 산출물을 입력으로 받아 한 단계씩 기능을 더해 나갑니다. 첫째 단계에서는 도메인 진입과 첫 도구를 만듭니다. 이는 지금 학습하고 있는 본 노트북에 해당합니다. 둘째 단계에서는 한국 설계기준과 재료 물성치를 리소스로 등록합니다. 셋째 단계에서는 다섯 번째 주차에서 만든 검색 보강 생성 체인을 리소스 백엔드로 통합합니다. 넷째 단계에서는 마이다스 시빌 입력 파일 파서를 도구로 추가합니다. 다섯째 단계에서는 시각적 검증 도구를 사용하여 서버를 점검합니다. 여섯째 단계에서는 명령줄 도구에 최종 등록하여 일반 대화에서 자연어로 호출할 수 있게 합니다. 마지막으로 통합본 노트북에서 모든 단계를 한 노트북 안에 압축하여 정리합니다.
